In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cinedata_medallion"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA gold")

print(f"catalog: {catalog}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
display(spark.sql(f"SHOW TABLES IN {gold_schema}"))

fact = spark.table(f"{gold_schema}.fact_movies_performance")

dim_movies = spark.table(f"{gold_schema}.dim_movies")
dim_genres = spark.table(f"{gold_schema}.dim_genres")
dim_people  = spark.table(f"{gold_schema}.dim_people")
dim_companies = spark.table(f"{gold_schema}.dim_companies")
dim_reviews  = spark.table(f"{gold_schema}.dim_reviews")

bridge_movie_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
bridge_movie_person = spark.table(f"{gold_schema}.bridge_movie_person")
bridge_movie_company = spark.table(f"{gold_schema}.bridge_movie_company")


In [0]:
# 1. Qual é a receita total (em R$) somada de todos os filmes da base?
df_receita_total = fact.agg(
    F.sum("receita_brl").alias("receita_total_brl"),
    F.count("receita_brl").alias("filmes_com_receita"),
    F.count("*").alias("filmes_na_fato"),
)

display(df_receita_total)

In [0]:
# 2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.
df_fato_filme = fact.join(dim_movies, on="sk_movie_id", how="inner")

df_popularidade = (
    df_fato_filme.select("titulo", "popularidade")
    .orderBy(F.desc("popularidade"))
    .limit(5)
)

display(df_popularidade)

In [0]:
#3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.
df_fato_genero = df_fato_filme.join(bridge_movie_genre, on="sk_movie_id", how="inner").join(dim_genres, on="sk_genre_id", how="inner")


df_qtd_por_genero = (
    df_fato_genero    
    .groupBy("genero")
    .agg(F.count("genero").alias("qtd"))
    .orderBy(F.desc("qtd"))
)

display(df_qtd_por_genero)

In [0]:
#4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (RANK())
df_fato_ranking = (
    df_fato_filme
    .select("titulo", "receita_usd", "receita_brl")
    .orderBy(F.desc("receita_brl"))
    .limit(10)
)

df_ranking = (
    df_fato_ranking
    .withColumn("posicao", F.rank().over(Window.orderBy(F.desc("receita_brl"))))
)

display(df_ranking)

In [0]:
# data limite superior: lançamento mais recente na base
# (apenas filmes lançados, ignorando datas futuras)
data_ref = (
    dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .filter(F.col("data_lancamento") <= F.current_date())
    .agg(F.max("data_lancamento"))
    .first()[0]
)

print(f"data de referência: {data_ref}")

In [0]:
#5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?
df_fato_atores = (
    df_fato_filme
    .join(bridge_movie_person, on="sk_movie_id", how="inner")
    .join(dim_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
       .filter(F.col("data_lancamento").between(
        F.add_months(F.lit(data_ref), -24),
        F.lit(data_ref),
    ))
)

#display(df_fato_atores)


df_qtd_atores = (
    df_fato_atores
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
)

# ranking - empatados recebem a mesma posição
janela_ranking = Window.orderBy(F.desc("qtd_filmes"))

df_top_atores = (
    df_qtd_atores
    .withColumn("posicao", F.dense_rank().over(janela_ranking))
    .filter(F.col("posicao") == 1)
    .select("nome_pessoa", "qtd_filmes")
)

display(df_top_atores)


In [0]:
#6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?

df_fato_produtoras = (
    df_fato_filme
    .filter(F.col("data_lancamento").between(
        F.add_months(F.lit(data_ref), -60),
        F.lit(data_ref),
    ))
    .join(bridge_movie_company, on="sk_movie_id", how="inner")
    .join(dim_companies, on="sk_company_id", how="inner")
)

df_lucro_produtoras = (
    df_fato_produtoras
    .groupBy("sk_company_id", "nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd"),
        F.sum("lucro_brl").alias("lucro_total_brl"),
        F.count("lucro_usd").alias("filmes_com_lucro"),
    )
    .filter(F.col("lucro_total_usd").isNotNull())
)

janela_ranking = Window.orderBy(F.desc("lucro_total_usd"))

df_top_produtora = (
    df_lucro_produtoras
    .withColumn("posicao", F.dense_rank().over(janela_ranking))
    .filter(F.col("posicao") == 1)
    .select("nome_produtora", "lucro_total_usd", "lucro_total_brl", "filmes_com_lucro")
)

display(df_top_produtora)